# CS781 Phase 1 v2 — better GPT-2-family language model (architecture ablation + a training recipe that fixes the original run's problems)

## What the original 3,000-step run did (read from your Phase 2 notebook, cells 0-16)
* `GPT2LMHeadModel` initialised **randomly**: 12 layers, hidden 768, 12 heads, n_inner 3072, **4,096 positions**, dropout 0, fp16 AMP, gradient checkpointing on.
* AdamW, batch 1 x accumulation 8, constant LR **configured as 5e-5**. Every document becomes `[BOS] tokens [EOS]`, is cut into 4,096-token chunks and the **last chunk is padded** (loss masked), so short judgments waste most of a sequence.
* The training stream walks the manifest **in file order (no shuffle)** and every restart begins again at index 0. The manifests appear to be sorted by domain with *acts first*; cell 2 below measures what the run could have seen.
* On resume the code calls `optimizer.load_state_dict(...)`, which **restores the saved learning rate (1e-5) and overrides the configured 5e-5**. That matches the checkpoint (lr 1e-5) and a log that starts at step 1001.
* There is no validation loop, so no held-out loss was ever recorded.

## What this notebook does
1. **Audit** (cell 2, read-only): is the manifest domain-sorted, and what would 3,000 steps have consumed?
2. **Fixed data pipeline**: domain-weighted, shuffled, packed without padding, validation/test documents never trained on, fixed held-out validation blocks per domain.
3. **Architecture presets, all `GPT2LMHeadModel`**: the original shape with 1,024 positions, a deeper/narrower 83M, a smaller 60M, a shallow-wide 68M (all within the 50M-300M Phase 1 range, head size 64, n_inner = 4 x hidden, so the Phase 3 loader reads them unchanged).
4. **Equal-GPU-time ablation** (about 15 min per preset), including a warm start from the old weights, so "better" is measured as lower held-out loss for the same compute, not guessed.
5. **Main run** with a fixed recipe (peak LR 6e-4, beta2 0.95, weight decay 0.1 on matrices, 131k tokens per step, warmup-stable-decay) and automatic resume; **export** with the registry line for Phase 3.

**Run order:** 1 setup -> 2 audit -> 3 data -> 4 presets -> 5 baselines -> 6 trainer -> 7 smoke test -> 8 ablation -> 9 main run -> 10 export.

## Read before running
* Check the spec on whether a re-trained Phase 1 model may be used in later phases; this notebook does not decide that.
* GPU quota is shared with every other notebook. Cell 7 prints how many steps fit in N GPU-hours.
* The data-side logic, the parameter formula, the audit and the schedule were tested on synthetic data. The model and training cells need your GPU: cell 7 is their first real test.


In [ ]:
# ================= CELL 1 — setup, paths, configuration =================
import os, sys, re, json, math, time, random, glob, gzip, gc, copy, bisect
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

BASE     = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"          # dedup corpus, manifests, tokenizer
RECOVERY = "/kaggle/input/datasets/oipranesh/cs781-phase4-final-recovery"          # holds indexes/source_index.jsonl (byte offsets)
P1_CKPT  = "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt"                # the 3,000-step model
P4_CKPT  = "/kaggle/input/datasets/oipranesh/phase4-step1000-verified-model/checkpoint_final.pt"            # the Phase 4 step-1000 model
OUT = Path("/kaggle/working/pretrain_v2"); OUT.mkdir(parents=True, exist_ok=True)
for p in (BASE, P1_CKPT): assert os.path.exists(p), f"missing: {p}"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); assert device.type == "cuda", "pretraining needs a GPU"
cap = torch.cuda.get_device_capability(0); USE_BF16 = cap[0] >= 8          # T4 (7.5) has no native bf16 -> fp16 + GradScaler
print("GPU:", torch.cuda.get_device_name(0), "| capability", cap, "| precision:", "bf16" if USE_BF16 else "fp16 + GradScaler")

DOMAINS = ["supreme_court", "tribunals", "high_courts", "acts"]
CTX = 1024                      # context length of the new models (downstream tasks use <= 512 tokens)
CFG = dict(
    preset="A_110M_L12_D768",   # see cell 4; suffix "__warm" starts from the 3,000-step weights (positions truncated to 1024)
    seq_len=1024, micro_bs=8, accum=16,              # 131,072 tokens per optimizer step
    lr_peak=6e-4, lr_min_frac=0.1, warmup=100, total_steps=2600, decay_start=None,   # warmup-stable-decay (decay over the last 30% by default)
    wd=0.1, betas=(0.9, 0.95), clip=1.0, eval_every=200, eval_tokens_per_domain=262144,
    domain_w={"supreme_court": 0.30, "tribunals": 0.30, "high_courts": 0.35, "acts": 0.05},   # downstream texts are judgments, not statutes
    min_words=50, seed=1234, run="v2_main")
print({k: v for k, v in CFG.items() if k != "domain_w"})


In [ ]:
# ================= CELL 2 — AUDIT (read-only, ~2-3 min): what did the original 3,000-step run actually see? =================
# The original loop walked the training manifest in file order (no shuffle) and every restart began at manifest index 0.
# If the manifest is sorted by domain (acts first), the model saw almost only statutes. This cell measures it.
MAN = f"{BASE}/manifests/train_manifest.jsonl"; IDX = f"{RECOVERY}/indexes/source_index.jsonl"
dom_re = re.compile(r'"domain":\s*"([^"]+)"'); hash_re = re.compile(r'"text_hash":\s*"([0-9a-f]{64})"')
N_FIRST = 60000; doms, first_hashes = [], []
with open(MAN, encoding="utf-8") as f:
    for i, line in enumerate(f):
        m = dom_re.search(line); doms.append(sys.intern(m.group(1)) if m else "?")
        if i < N_FIRST: first_hashes.append(hash_re.search(line).group(1))
runs, s = [], 0
for i in range(1, len(doms) + 1):
    if i == len(doms) or doms[i] != doms[s]: runs.append((doms[s], s, i - 1)); s = i
print(f"manifest lines: {len(doms):,} | consecutive same-domain runs: {len(runs):,}")
if len(runs) < 50: print("  => the manifest is SORTED by domain:", [(d, f"{a:,}-{b:,}") for d, a, b in runs])
else:               print("  => the manifest looks shuffled/mixed; first runs:", runs[:6])
for k in (8000, 24000, 60000): print(f"  domain mix of the first {k:,} manifest lines:", dict(Counter(doms[:k])))

BYTES_PER_TOKEN = 4.2    # ASSUMPTION (typical for English BPE); only used to estimate how many 4,096-token chunks each document yields
wanted, blen = set(first_hashes), {}
with open(IDX, encoding="utf-8") as f:
    for line in f:
        h = line[14:78]
        if h in wanted: blen[h] = json.loads(line)["byte_length"]
        if len(blen) == len(wanted): break
cum, tot = [], 0
for h in first_hashes: tot += max(1, math.ceil((blen.get(h, 4096 * BYTES_PER_TOKEN) / BYTES_PER_TOKEN + 2) / 4096)); cum.append(tot)
POINTS = (("Phase 4 steps 1-1000   (8,000 sequences)", 8000), ("steps 1001-3000 if restarted at the top (16,000)", 16000), ("3,000 steps from the top (24,000)", 24000))
print("\nEstimated consumption (each optimizer step = 8 sequences of 4,096 tokens, padded):")
for label, nseq in POINTS:
    if cum[-1] < nseq: print(f"  {label:52s} -> not reached within the first {len(cum):,} documents (increase N_FIRST)"); continue
    k = bisect.bisect_left(cum, nseq) + 1
    print(f"  {label:52s} -> first {k:6,} documents | mix: {dict(Counter(doms[:k]))}")


In [ ]:
# ================= CELL 3 — streaming corpus: held-out split, domain-weighted shuffled mix, packing WITHOUT padding =================
from tokenizers import Tokenizer
tok = Tokenizer.from_file(f"{BASE}/tokenizer/legal_bpe_32k.json")
BOS, EOS, PAD = tok.token_to_id("[BOS]"), tok.token_to_id("[EOS]"), tok.token_to_id("[PAD]")
V = tok.get_vocab_size(); assert V == 32000 and None not in (BOS, EOS, PAD); print("tokenizer vocab:", V, "| BOS/EOS/PAD:", BOS, EOS, PAD)

def shard_files(domain):
    fs = sorted(glob.glob(f"{BASE}/dedup/{domain}/*.jsonl") + glob.glob(f"{BASE}/dedup/{domain}/*.jsonl.gz")); assert fs, f"no shards for {domain}"; return fs
def open_text(p): return gzip.open(p, "rt", encoding="utf-8") if p.endswith(".gz") else open(p, encoding="utf-8")
def load_hash_set(split):
    s = set()
    with open(f"{BASE}/manifests/{split}_manifest.jsonl", encoding="utf-8") as f:
        for line in f: s.add(json.loads(line)["text_hash"])
    return s
HELD = {sp: load_hash_set(sp) for sp in ("validation", "test")}
SKIP = HELD["validation"] | HELD["test"]               # training never sees validation or test documents
print({k: f"{len(v):,} docs" for k, v in HELD.items()}, "| shards:", {d: len(shard_files(d)) for d in DOMAINS})

def iter_docs(domain, rng, skip, min_words, files=None):
    files = list(files or shard_files(domain))
    while True:
        rng.shuffle(files)
        for p in files:
            with open_text(p) as f:
                for line in f:
                    try: r = json.loads(line)
                    except Exception: continue
                    if r.get("text_hash") in skip: continue
                    t = r.get("text")
                    if not t: continue
                    try: wc = int(r.get("word_count"))
                    except Exception: wc = len(t.split())
                    if wc >= min_words: yield t

def shuffled(it, buf, rng):                              # shard files are ordered, so mix them through a buffer
    pool = []
    for x in it:
        pool.append(x)
        if len(pool) >= buf:
            i = rng.randrange(len(pool)); pool[i], pool[-1] = pool[-1], pool[i]; yield pool.pop()

class PackedStream:
    """domain-weighted documents -> [BOS] tokens [EOS] -> packed into [micro_bs, seq_len] blocks with NO padding"""
    def __init__(self, tokenizer, cfg, seed, skip):
        self.tok, self.cfg, self.rng = tokenizer, cfg, random.Random(seed); self.buf = []; self.dom_tokens = Counter()
        self.gens = {d: shuffled(iter_docs(d, random.Random(seed + i), skip, cfg["min_words"]), 1000, random.Random(seed + 100 + i)) for i, d in enumerate(DOMAINS)}
        self.w = [cfg["domain_w"][d] for d in DOMAINS]
    def _refill(self, need):
        while len(self.buf) < need:
            ds = [self.rng.choices(DOMAINS, weights=self.w)[0] for _ in range(32)]; texts = [next(self.gens[d]) for d in ds]
            for d, enc in zip(ds, self.tok.encode_batch(texts, add_special_tokens=False)):
                ids = [BOS] + enc.ids + [EOS]; self.buf.extend(ids); self.dom_tokens[d] += len(ids)
    def next_batch(self):
        B, T = self.cfg["micro_bs"], self.cfg["seq_len"]; self._refill(B * T)
        x = torch.tensor(self.buf[:B * T], dtype=torch.long).view(B, T); del self.buf[:B * T]; return x

def build_val_blocks(cfg, cache=OUT / "val_blocks.pt"):
    """fixed held-out blocks per domain, from validation-manifest documents only, spread over several shards"""
    if cache.exists(): return torch.load(cache)
    T = cfg["seq_len"]; per = cfg["eval_tokens_per_domain"] // T; need = per * T; val = {}
    for d in DOMAINS:
        files = shard_files(d); idx = sorted({int(i * (len(files) - 1) / max(1, min(6, len(files)) - 1)) for i in range(min(6, len(files)))})
        quota = math.ceil(need / len(idx)); toks = []
        for fi in idx:
            got = 0
            with open_text(files[fi]) as f:
                for line in f:
                    try: r = json.loads(line)
                    except Exception: continue
                    if r.get("text_hash") in HELD["validation"] and r.get("text"):
                        ids = [BOS] + tok.encode(r["text"], add_special_tokens=False).ids + [EOS]; toks += ids; got += len(ids)
                        if got >= quota: break
        assert len(toks) >= need, f"{d}: only {len(toks)} validation tokens found (need {need})"
        val[d] = torch.tensor(toks[:need], dtype=torch.long).view(per, T)
    torch.save(val, cache); return val
VAL = build_val_blocks(CFG); print("validation blocks:", {d: tuple(v.shape) for d, v in VAL.items()}, "| tokens:", sum(v.numel() for v in VAL.values()))


In [ ]:
# ================= CELL 4 — GPT-2-family architecture presets (all built with Hugging Face GPT2LMHeadModel) =================
from transformers import GPT2Config, GPT2LMHeadModel
def gpt2_params(L, d, vocab=32000, ctx=CTX):           # closed form for GPT-2 with tied head and n_inner = 4*d
    inner = 4 * d; per = 2 * d + (d * 3 * d + 3 * d) + (d * d + d) + 2 * d + (d * inner + inner) + (inner * d + d)
    return vocab * d + ctx * d + L * per + 2 * d
assert gpt2_params(12, 768, 32000, 4096) == 112_777_728, "formula must reproduce your checkpoint's parameter count"

PRESETS = {   # depth / width / size trade-offs; every preset keeps head size 64 and n_inner = 4*d (so the Phase 3 loader works unchanged)
    "A_110M_L12_D768": dict(n_layer=12, n_embd=768, n_head=12),    # same shape as the original model, but 1024 positions
    "B_83M_L16_D576":  dict(n_layer=16, n_embd=576, n_head=9),     # deeper and narrower
    "C_60M_L8_D640":   dict(n_layer=8,  n_embd=640, n_head=10),    # smaller: ~1.8x cheaper per token
    "E_68M_L6_D768":   dict(n_layer=6,  n_embd=768, n_head=12),    # shallow and wide: fewest sequential layers
}
def parse_preset(name): base, _, suf = name.partition("__"); assert base in PRESETS, name; return base, ("warm3000" if suf == "warm" else "scratch")
print(f"{'preset':18s}{'params':>9s}{'non-emb':>9s}{'rel. cost/token':>17s}   (Phase 1 requires 50M-300M parameters)")
ref = None
for k, p in PRESETS.items():
    n = gpt2_params(p["n_layer"], p["n_embd"]); ne = n - 32000 * p["n_embd"] - CTX * p["n_embd"]; cost = 6 * (ne + 32000 * p["n_embd"]); ref = ref or cost
    assert 50e6 <= n <= 300e6; print(f"{k:18s}{n/1e6:8.1f}M{ne/1e6:8.1f}M{cost/ref:17.2f}")

def _make(cls, cfgobj):
    try: return cls._from_config(cfgobj, attn_implementation="sdpa")
    except Exception: return cls(cfgobj)

def build_model(name):
    base, init = parse_preset(name); p = PRESETS[base]; L, d, H = p["n_layer"], p["n_embd"], p["n_head"]
    gc_ = GPT2Config(vocab_size=V, n_positions=CTX, n_ctx=CTX, n_embd=d, n_layer=L, n_head=H, n_inner=4 * d, resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0,
                     bos_token_id=BOS, eos_token_id=EOS, pad_token_id=PAD, use_cache=False)
    m = _make(GPT2LMHeadModel, gc_); n = sum(q.numel() for q in m.parameters())
    assert 50e6 <= n <= 300e6 and abs(n - gpt2_params(L, d)) < 0.001 * n, f"parameter count {n:,} unexpected"
    if init == "warm3000":
        assert (L, d, H) == (12, 768, 12), "warm start is only defined for the original shape (preset A)"
        ck = torch.load(P1_CKPT, map_location="cpu", weights_only=False); sd = ck["model_state_dict"]
        sd["transformer.wpe.weight"] = sd["transformer.wpe.weight"][:CTX].clone(); print("warm start:", m.load_state_dict(sd, strict=True)); del ck, sd
    m.config.use_cache = False
    mc = dict(family="gpt2", vocab_size=V, context_length=CTX, hidden_size=d, num_layers=L, num_heads=H, n_inner=4 * d)
    return m.to(device), mc, n

def amp_ctx(): return torch.autocast("cuda", dtype=torch.bfloat16 if USE_BF16 else torch.float16)
@torch.no_grad()
def evaluate(model, val, bs=8):
    model.eval(); res = {}
    for d, blocks in val.items():
        tot, n = 0.0, 0
        for i in range(0, len(blocks), bs):
            x = blocks[i:i + bs].to(device)
            with amp_ctx(): logits = model(input_ids=x).logits
            tot += F.cross_entropy(logits[:, :-1].float().reshape(-1, logits.shape[-1]), x[:, 1:].reshape(-1), reduction="sum").item(); n += x[:, 1:].numel()
        res[d] = tot / n
    res["overall"] = float(np.mean([res[d] for d in val])); model.train(); return res
def fmt(res): return " | ".join(f"{k} {v:.3f}" for k, v in res.items())


In [ ]:
# ================= CELL 5 — baselines: the EXISTING checkpoints on held-out text, per domain (read-only, ~3 min) =================
def load_old(path):
    gc_ = GPT2Config(vocab_size=V, n_positions=4096, n_ctx=4096, n_embd=768, n_layer=12, n_head=12, n_inner=3072, resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0)
    m = _make(GPT2LMHeadModel, gc_).to(device); ck = torch.load(path, map_location="cpu", weights_only=False, mmap=True)
    try: m.load_state_dict(ck.get("model_state_dict", ck), strict=True)
    except Exception as e: print("   could not load as GPT-2:", str(e)[:160]); return None
    return m
BASELINE = {}
for name, path in (("phase1_step3000", P1_CKPT), ("phase4_step1000", P4_CKPT)):
    if not os.path.exists(path): print(name, "missing, skipped"); continue
    m = load_old(path)
    if m is None: continue
    BASELINE[name] = evaluate(m, VAL); print(f"{name:16s} val nats/token -> {fmt(BASELINE[name])}  (overall perplexity {math.exp(BASELINE[name]['overall']):.0f})")
    del m; gc.collect(); torch.cuda.empty_cache()
json.dump(BASELINE, open(OUT / "baseline.json", "w"))
if "phase1_step3000" in BASELINE:
    b = BASELINE["phase1_step3000"]; others = np.mean([b[d] for d in ("supreme_court", "tribunals", "high_courts")])
    print(f"\nacts loss {b['acts']:.2f} vs judgment domains {others:.2f}: " + ("acts are far easier => consistent with the model having trained mostly on statutes." if b["acts"] < others - 0.5 else "no large acts advantage."))
print("uniform-guess loss would be", round(math.log(V), 2), "nats. Every new run is judged against these numbers.")


In [ ]:
# ================= CELL 6 — trainer (warmup-stable-decay, fp16/bf16, resume, validation) + equal-time architecture ablation =================
def lr_at(step, c):
    w, T = c["warmup"], c["total_steps"]; ds = c["decay_start"] or int(0.7 * T); lo = c["lr_peak"] * c["lr_min_frac"]
    if step < w: return c["lr_peak"] * (step + 1) / w
    if step < ds: return c["lr_peak"]
    return c["lr_peak"] + (lo - c["lr_peak"]) * min(1.0, (step - ds) / max(1, T - ds))

def train(cfg, steps=None, accum=None, do_eval=True, save=True, resume=True, minutes_budget=None, eval_every=None, keep_model=False, quiet=False):
    c = {**cfg}; c["accum"] = accum or c["accum"]; rdir = OUT / c["run"]; rdir.mkdir(exist_ok=True)
    torch.manual_seed(c["seed"]); model, mc, n_params = build_model(c["preset"]); model.train()
    decay = [p for p in model.parameters() if p.ndim >= 2]; nodecay = [p for p in model.parameters() if p.ndim < 2]
    groups = [{"params": decay, "weight_decay": c["wd"]}, {"params": nodecay, "weight_decay": 0.0}]
    try: opt = torch.optim.AdamW(groups, lr=c["lr_peak"], betas=c["betas"], eps=1e-8, fused=True)
    except Exception: opt = torch.optim.AdamW(groups, lr=c["lr_peak"], betas=c["betas"], eps=1e-8)
    scaler = torch.amp.GradScaler("cuda", enabled=not USE_BF16); step, tokens = 0, 0; ck_path = rdir / "latest.pt"
    if save and resume and ck_path.exists():
        ck = torch.load(ck_path, map_location="cpu", weights_only=False); model.load_state_dict(ck["model_state_dict"]); opt.load_state_dict(ck["optimizer_state_dict"])
        scaler.load_state_dict(ck["scaler_state_dict"]); step, tokens = ck["step"], ck["tokens"]; print(f"resumed from step {step}"); del ck
        for g in opt.param_groups: g["lr"] = lr_at(step, c)       # (optimizer.load_state_dict restores the OLD lr; we always set it explicitly)
    stream = PackedStream(tok, c, c["seed"] + step, SKIP); T = c["seq_len"] * c["micro_bs"] * c["accum"]
    end = steps if steps is not None else c["total_steps"]; log = open(rdir / "log.jsonl", "a"); t0 = time.time(); last_save = time.time()
    base = json.load(open(OUT / "baseline.json")) if (OUT / "baseline.json").exists() else {}; b_overall = base.get("phase1_step3000", {}).get("overall")
    best, run_loss, nb, step_times, hist = float("inf"), 0.0, 0, [], []; ev = eval_every or c["eval_every"]
    while step < end:
        ts = time.time(); lr = lr_at(step, c)
        for g in opt.param_groups: g["lr"] = lr
        for _ in range(c["accum"]):
            x = stream.next_batch().to(device)
            with amp_ctx(): loss = model(input_ids=x, labels=x).loss
            if not torch.isfinite(loss): print("non-finite loss, skipping microbatch"); continue
            scaler.scale(loss / c["accum"]).backward(); run_loss += loss.item(); nb += 1
        scaler.unscale_(opt); gn = torch.nn.utils.clip_grad_norm_(model.parameters(), c["clip"]); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        step += 1; tokens += T; torch.cuda.synchronize(); step_times.append(time.time() - ts)
        if step % 10 == 0:
            rec = dict(step=step, loss=run_loss / max(nb, 1), lr=lr, grad_norm=float(gn), tokens=tokens, tok_s=T / float(np.median(step_times[-10:])), elapsed_min=(time.time() - t0) / 60)
            run_loss, nb = 0.0, 0; log.write(json.dumps(rec) + "\n"); log.flush()
            if not quiet and (step % 50 == 0 or step <= 20): print(f"[{c['run']}] step {step:5d} | loss {rec['loss']:.3f} | lr {lr:.2e} | gnorm {rec['grad_norm']:.2f} | {rec['tok_s']/1e3:.1f}k tok/s | {rec['elapsed_min']:.1f} min | {tokens/1e6:.0f}M tokens", flush=True)
        if do_eval and (step % ev == 0 or step == end):
            r = evaluate(model, VAL); hist.append(dict(step=step, tokens=tokens, elapsed_min=(time.time() - t0) / 60, val=r)); log.write(json.dumps(dict(step=step, val=r)) + "\n"); log.flush()
            if not quiet: print(f"   VALIDATION step {step}: {fmt(r)}" + (f" | vs 3000-step baseline {r['overall']-b_overall:+.3f}" if b_overall else ""), flush=True)
            if save and r["overall"] < best: best = r["overall"]; torch.save({"model_state_dict": model.state_dict(), "model_config": mc, "step": step, "val": r}, rdir / "best.pt")
        if save and time.time() - last_save > 20 * 60:
            torch.save({"model_state_dict": model.state_dict(), "optimizer_state_dict": opt.state_dict(), "scaler_state_dict": scaler.state_dict(), "step": step, "tokens": tokens, "model_config": mc, "cfg": c}, ck_path)
            last_save = time.time(); print("   checkpoint saved")
        if minutes_budget and (time.time() - t0) / 60 > minutes_budget: print(f"time budget reached at step {step}"); break
    if save:
        torch.save({"model_state_dict": model.state_dict(), "optimizer_state_dict": opt.state_dict(), "scaler_state_dict": scaler.state_dict(), "step": step, "tokens": tokens, "model_config": mc, "cfg": c}, ck_path)
        torch.save({"model_state_dict": model.state_dict(), "model_config": {**mc, "trained_steps": step, "tokens": tokens}, "step": step}, rdir / "final_model.pt")
    log.close(); info = dict(history=hist, tok_s=float(T / np.median(step_times[2:] if len(step_times) > 3 else step_times)), params=n_params, steps=step, tokens=tokens,
                             minutes=(time.time() - t0) / 60, mix={d: v / max(1, sum(stream.dom_tokens.values())) for d, v in stream.dom_tokens.items()}, model=model if keep_model else None)
    del opt, scaler; gc.collect(); torch.cuda.empty_cache(); return info

def plan_budget(tok_s, hours, cfg=CFG):
    per = cfg["seq_len"] * cfg["micro_bs"] * cfg["accum"]; steps = int(hours * 3600 * tok_s / per)
    print(f"at {tok_s/1e3:.1f}k tok/s, {hours} GPU-hours = {steps} steps = {steps*per/1e9:.2f}B tokens  -> set CFG['total_steps'] = {steps}")

def ablate(names, minutes=12, tokens_per_step=65536):
    """Give every preset the SAME wall-clock budget and the SAME data order; the LR schedule is sized to each run, so each finishes fully decayed."""
    rows = []
    for nm in names:
        probe = train({**CFG, "preset": nm, "run": "probe"}, steps=8, accum=2, do_eval=False, save=False, resume=False, quiet=True); tps = probe["tok_s"]
        steps = max(50, int(minutes * 60 * tps / tokens_per_step) // 10 * 10); apr = tokens_per_step // (CFG["seq_len"] * CFG["micro_bs"])
        c = {**CFG, "preset": nm, "run": f"abl_{nm}", "total_steps": steps, "warmup": max(10, steps // 20), "decay_start": int(0.7 * steps), "eval_every": max(25, steps // 6)}
        print(f"\n=== {nm}: {tps/1e3:.1f}k tok/s -> {steps} steps of {tokens_per_step:,} tokens in ~{minutes} min ===")
        info = train(c, steps=steps, accum=apr, do_eval=True, save=False, resume=False); last = info["history"][-1]
        rows.append(dict(preset=nm, params_M=info["params"] / 1e6, tok_per_s=tps, steps=steps, tokens_M=info["tokens"] / 1e6, minutes=info["minutes"], **{f"val_{k}": v for k, v in last["val"].items()}))
    df = pd.DataFrame(rows).sort_values("val_overall"); df.to_csv(OUT / "ablation.csv", index=False)
    print("\n=== RESULT (same GPU time per preset; lower val_overall is better) ==="); print(df.round(3).to_string(index=False)); return df


In [ ]:
# ================= CELL 7 — SMOKE TEST (12 optimizer steps, ~2 min): data -> step -> validation -> throughput =================
info = train({**CFG, "run": "smoke"}, steps=12, accum=2, do_eval=True, save=False, resume=False)
print(f"\nmeasured throughput for {CFG['preset']}: {info['tok_s']/1e3:.1f}k tokens/s")
for h in (4, 6, 8, 10): plan_budget(info["tok_s"], h)
del info; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ================= CELL 8 — ARCHITECTURE ABLATION (~15 min per preset; equal GPU time; same data order) =================
# Includes the 3,000-step warm start ("__warm") to test whether the old weights help or hurt. Compare against the baselines from cell 5.
ABL = ablate(["A_110M_L12_D768", "A_110M_L12_D768__warm", "B_83M_L16_D576", "C_60M_L8_D640", "E_68M_L6_D768"], minutes=12)
print("\nThe winner here is a decision aid, not proof: 12-minute runs favour smaller models, and ranks can change with more tokens. "
      "If the top two are within ~0.03 nats, pick the larger one (it has more headroom for a longer run).")


In [ ]:
# ================= CELL 9 — MAIN RUN with the chosen preset (resumes automatically; checkpoint every 20 min) =================
# 1) set CFG['preset'] from cell 8, 2) set CFG['total_steps'] from cell 7's budget plan, 3) run.
# GATE: by the 2nd validation line, overall loss must be clearly below the 3,000-step baseline and still falling; if not, stop and report.
CFG["run"] = "v2_main"
info = train(CFG)       # set minutes_budget=<n> to stop (and save) after n minutes


In [ ]:
# ================= CELL 10 — export: files to use downstream + registry lines for the Phase 3 notebook =================
for run in sorted(p.name for p in OUT.iterdir() if p.is_dir() and (p / "final_model.pt").exists()):
    ck = torch.load(OUT / run / "final_model.pt", map_location="cpu", weights_only=False, mmap=True); mc = ck["model_config"]
    print(f"{run}: {OUT/run/'final_model.pt'} ({os.path.getsize(OUT/run/'final_model.pt')/2**20:.0f} MB) | step {ck['step']} | {mc}")
    print(f'   Phase 3 notebook (cell 7) ->  ARCHS["{run}"] = dict(path="{OUT/run/"final_model.pt"}", family="gpt2")')
    vals = [json.loads(l) for l in open(OUT / run / "log.jsonl") if '"val"' in l]
    if vals: print("   validation history (overall nats/token):", [(v["step"], round(v["val"]["overall"], 3)) for v in vals])
    del ck
print("\nKeys follow the original layout (transformer.*), and n_inner = 4*hidden, so the existing Phase 3 loader reads these files unchanged.")
